# PathSnap model training on Colab

This notebook prepares candidate features from raw `.track`, `.nodes`, `.arcs`, and `.route` trips, caches those rows as CSV, then trains/evaluates Member 1's MLP and KNN. The MLP uses a compact five-feature input by default. The CSV is an intermediate cache, not a required source format. It reads only `train` and `student_test`; it never opens evaluator trips.

Set `QUICK_MODE = False` for the full 80-trip train / 10-trip student-test run. Quick mode is for debugging only. The raw-route alignment assumption is documented in the source; inspect candidate misses and alignment before using any results in the report.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, csv, shutil, subprocess, time, json

DATA_ROOT = Path('/content/drive/MyDrive/PathSnap/datasets')  # contains train/ and student_test/
SOURCE_DIR = Path('/content/drive/MyDrive/PathSnap/source')  # optional source folder
OUT_ROOT = Path('/content/drive/MyDrive/PathSnap/colab_results')  # persistent Drive outputs
QUICK_MODE = False  # True uses 5 train trips, 2 student-test trips, and small model configs
print('Data root:', DATA_ROOT, 'exists:', DATA_ROOT.exists())


## Make project code available

The current implementation may be newer than the public `main` branch. Put these files in `SOURCE_DIR` (or upload them when prompted): `data_processing/candidate_features.py`, `models/mlp.py`, and `models/knn.py`.


In [ ]:
repo=Path('/content/pathsnap-src')
for sub in ('data_processing','models'):
    (repo/sub).mkdir(parents=True,exist_ok=True)
source_files={
 'candidate_features.py':repo/'data_processing/candidate_features.py',
 'mlp.py':repo/'models/mlp.py',
 'knn.py':repo/'models/knn.py',
}
for name,dst in source_files.items():
    candidates=[SOURCE_DIR/name, SOURCE_DIR/'data_processing'/name, SOURCE_DIR/'models'/name]
    src=next((p for p in candidates if p.exists()),None)
    if src: shutil.copy2(src,dst)
missing=[n for n,p in source_files.items() if not p.exists()]
if missing:
    from google.colab import files
    uploaded=files.upload()
    for name,data in uploaded.items():
        if name in source_files: source_files[name].write_bytes(data)
if any(not p.exists() for p in source_files.values()):
    raise FileNotFoundError('Upload candidate_features.py, mlp.py, and knn.py')
sys.path.insert(0,str(repo))
try:
    import numpy, sklearn
except ImportError:
    subprocess.check_call([sys.executable,'-m','pip','install','numpy','scikit-learn','joblib'])


## Prepare candidate feature cache from raw trips

This streams trip-by-trip so the notebook does not hold every generated candidate row in memory at once. It uses a spatial index over route arcs only for route alignment and vectorized coordinate conversion. Set `REBUILD_FEATURE_CACHE = True` to regenerate; otherwise existing cache CSVs on Drive are reused.


In [ ]:
from data_processing.candidate_features import process_trip

if not DATA_ROOT.exists(): raise FileNotFoundError(f'Put datasets under {DATA_ROOT}')
train_count=5 if QUICK_MODE else 80
student_count=2 if QUICK_MODE else 10
selected={
 'train':sorted(p for p in (DATA_ROOT/'train').iterdir() if p.is_dir())[:train_count],
 'student_test':sorted(p for p in (DATA_ROOT/'student_test').iterdir() if p.is_dir())[:student_count],
}
if len(selected['train']) < (3 if QUICK_MODE else 5): raise ValueError('Not enough training trips for requested grouped CV')
feature_dir=OUT_ROOT/'features'; feature_dir.mkdir(parents=True,exist_ok=True)
REBUILD_FEATURE_CACHE = False  # Set True only when source/data/feature settings change
MODEL_FEATURES=['distance_m','distance_gap_m','heading_cos','gps_speed_mps','candidate_rank']
CACHE_COLUMNS=['trip_id','observation_id','arc_id',*MODEL_FEATURES,'is_match']
manifest_path=feature_dir/'cache_manifest.json'
expected_manifest={'trip_ids':{part:[folder.name for folder in folders] for part,folders in selected.items()},
                   'features':CACHE_COLUMNS,'candidate_k':8,'initial_radius_m':100,'max_radius_m':800}
cache_exists=(all((feature_dir/f'{part}_candidates.csv').exists() for part in selected)
              and manifest_path.exists()
              and json.loads(manifest_path.read_text())==expected_manifest)
if REBUILD_FEATURE_CACHE or not cache_exists:
    summary={}
    for part,folders in selected.items():
        out_csv=feature_dir/f'{part}_candidates.csv'
        total_miss=total_obs=0
        with out_csv.open('w',newline='') as f:
            writer=None
            for idx,folder in enumerate(folders,1):
                trip_rows,info=process_trip(folder)
                total_miss += info['candidate_misses']; total_obs += info['observations']
                if trip_rows:
                    if writer is None:
                        fieldnames=CACHE_COLUMNS; writer=csv.DictWriter(f,fieldnames=fieldnames); writer.writeheader()
                    writer.writerows({k:r[k] for k in CACHE_COLUMNS} for r in trip_rows)
                print(f"{part}: {idx}/{len(folders)} trip {folder.name}; observations={info['observations']}; candidate_misses={info['candidate_misses']}")
        summary[part]={'trips':len(folders),'observations':total_obs,'candidate_misses':total_miss,
                       'candidate_miss_rate':total_miss/max(total_obs,1),'csv':str(out_csv)}
        print(summary[part])
    (OUT_ROOT/'candidate_generation_summary.json').write_text(json.dumps(summary,indent=2))
    manifest_path.write_text(json.dumps(expected_manifest,indent=2))
    print('Feature cache complete:', feature_dir)
else:
    print('Reusing existing candidate cache:', feature_dir)


## Train MLP and KNN

Full mode uses 5 trip-grouped folds and the baseline settings documented with each model. MLP uses 15 epochs and large batches; KNN uses 15 distance-weighted neighbors. Quick mode reduces the folds/epochs/neighbors and must not be presented as the final experiment. Both scripts use the same cached feature columns and train/student-test partitions.


In [ ]:
folds=3 if QUICK_MODE else 5
epochs=3 if QUICK_MODE else 15
neighbors=5 if QUICK_MODE else 15
train_csv=feature_dir/'train_candidates.csv'; student_csv=feature_dir/'student_test_candidates.csv'

mlp_cmd=[sys.executable,str(repo/'models/mlp.py'),'--train',str(train_csv),'--student-test',str(student_csv),
         '--out',str(OUT_ROOT/'mlp'),'--folds',str(folds),'--epochs',str(epochs),'--batch-size','8192']
print('MLP:', ' '.join(mlp_cmd)); subprocess.run(mlp_cmd,check=True)

knn_cmd=[sys.executable,str(repo/'models/knn.py'),'--train',str(train_csv),'--student-test',str(student_csv),
         '--out',str(OUT_ROOT/'knn'),'--folds',str(folds),'--neighbors',str(neighbors),'--features',','.join(MODEL_FEATURES)]
print('KNN:', ' '.join(knn_cmd)); subprocess.run(knn_cmd,check=True)
print('MLP metrics:',(OUT_ROOT/'mlp/metrics.json').read_text())
print('KNN metrics:',(OUT_ROOT/'knn/metrics.json').read_text())


## Notes before reporting

- Full mode is the real development experiment; quick mode is only a smoke run.
- Training/CV uses only training trips; student-test trips are evaluated after fitting and may be used for development comparison as the guidelines allow. Never tune against evaluator trips.
- Review the route-to-GPS alignment assumption and candidate miss rates. A high candidate miss rate limits any classifier's achievable score.
- MLP/KNN outputs include metrics and saved artifacts; KNN has no epoch-loss curve. Store the notebook and outputs in Drive, since Colab `/content` is temporary.
- Runtime depends on candidate counts and Colab RAM/CPU. This MLP is NumPy-based and KNN uses scikit-learn CPU KD-trees; a GPU runtime does not automatically accelerate either model.
